<a href="https://colab.research.google.com/github/YuktiVasuja/Image2Playlist/blob/main/vibe_music_recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow

In [ ]:
!pip install spotipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.2/354.2 kB 13.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import numpy as np
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing import image

# Load VGG16 model
model = VGG16(weights='imagenet', include_top=False, pooling='avg')

def extract_image_features(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)
    features = model.predict(x)
    return features.flatten()


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("/content/drive/MyDrive/song_recommender_project/.env")




True

In [ ]:
sp = spotipy.Spotify(
    auth_manager=SpotifyClientCredentials(

        client_id = os.getenv("SPOTIPY_CLIENT_ID"),
        client_secret = os.getenv("SPOTIPY_CLIENT_SECRET")
    )
)


In [ ]:
vibe_to_features = {
    "calm":    {"seed_genres": "ambient", "target_energy": 0.2, "target_valence": 0.6},
    "energetic":{"seed_genres": "dance", "target_energy": 0.8, "target_valence": 0.7},
    "sad":     {"seed_genres": "acoustic", "target_energy": 0.2, "target_valence": 0.2},
    "chill":   {"seed_genres": "indie", "target_energy": 0.4, "target_valence": 0.5},
    "romantic":{"seed_genres": "romance", "target_energy": 0.5, "target_valence": 0.7}
}


In [ ]:
import pandas as pd
from sklearn.neighbors import NearestNeighbors


# Paths
DATASET_PATH = "/content/drive/MyDrive/song_recommender_project/dataset"


image_features = []
image_vibes = []

# Load dataset images
for vibe in os.listdir(DATASET_PATH):
    vibe_path = os.path.join(DATASET_PATH, vibe)
    if not os.path.isdir(vibe_path):
        continue

    for img_file in os.listdir(vibe_path):
        img_path = os.path.join(vibe_path, img_file)
        if os.path.isfile(img_path):  # Ensure it's a file, not a directory
            features = extract_image_features(img_path)
            image_features.append(features)
            image_vibes.append(vibe)

image_features = np.array(image_features)

# Fit KNN
knn = NearestNeighbors(n_neighbors=3, metric='cosine')
knn.fit(image_features)






1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 984ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 797ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 656ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 651ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 647ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 663ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 663ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step   
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 644ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step   
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 663ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 664ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 656ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 720ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 637ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 657ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 656ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 801ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 931ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 644ms/step
1/1 

NearestNeighbors(metric='cosine', n_neighbors=3)

In [ ]:
def recommend_songs(query_image_path):
    # 1. Extract image features
    query_features = extract_image_features(query_image_path).reshape(1, -1)

    # 2. Find nearest images
    distances, indices = knn.kneighbors(query_features)
    recommended_vibes = [image_vibes[i] for i in indices[0]]

    # 3. Majority vote for final vibe
    final_vibe = max(set(recommended_vibes), key=recommended_vibes.count)
    print(f"\nDetected vibe: {final_vibe.upper()}")

    # 4. Get Spotify genre mapping
    params = vibe_to_features.get(final_vibe)
    if params is None:
        print("No Spotify mapping found for this vibe.")
        return

    # 5. Use Spotify search for tracks (fallback for recommendations)
    query_genre = params["seed_genres"]
    results = sp.search(q=query_genre, type="track", limit=10)

    # 6. Print recommended tracks
    print("Recommended Songs (Spotify Search):")
    for track in results["tracks"]["items"]:
        song_name = track["name"]
        artist = track["artists"][0]["name"]
        preview = track["preview_url"]
        spotify_url = track["external_urls"]["spotify"]

        print(f"- {song_name} by {artist}")
        if preview:
            print(f"  Preview URL: {preview}")
        print(f"  Spotify URL: {spotify_url}\n")


In [ ]:
test_image = "/content/test3.jpeg"  # user image
recommend_songs(test_image)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 659ms/step

Detected vibe: CHILL
Recommended Songs (Spotify Search):
- End of Beginning by Djo
  Spotify URL: https://open.spotify.com/track/3qhlB30KknSejmIvZZLjOD

- Feathered Indians by Tyler Childers
  Spotify URL: https://open.spotify.com/track/2tgQaL85WoRfgEa4hFQgrE

- Sienna by The Marías
  Spotify URL: https://open.spotify.com/track/0InIeZW4P6VO7dUGRM4AKH

- Indie Rokkers by MGMT
  Spotify URL: https://open.spotify.com/track/1ByAA0Xb8uqmNpB4yQgvTi

- STILL HOLY by indie tribe
  Spotify URL: https://open.spotify.com/track/0nwrCHhxawj5mSHvb6UkpA

- No One Noticed by The Marías
  Spotify URL: https://open.spotify.com/track/3siwsiaEoU4Kuuc9WKMUy5

- 505 by Arctic Monkeys
  Spotify URL: https://open.spotify.com/track/58ge6dfP91o9oXMzq3XkIS

- Still Holy (feat. Ryan Ofei & Naomi Raine) by Tribl
  Spotify URL: https://open.spotify.com/track/4Cw36mfQ1z4JnjFOEtX4hP

- Indie by VIOLENT VIRA
  Spotify URL: https://open.spotify.com/track/0URBL54h1GTCBg6Y7Jj3HG

-